In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os

data_path = '/content/drive/MyDrive/skin-cancer-fusion/data'
print("Contents of your data folder:")
for item in os.listdir(data_path):
    print(" -", item)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Contents of your data folder:
 - imgs_part_2.zip
 - imgs_part_3.zip
 - imgs_part_1.zip
 - metadata.csv


In [3]:
import zipfile
import os

data_path = '/content/drive/MyDrive/skin-cancer-fusion/data'
extract_path = '/content/images'

os.makedirs(extract_path, exist_ok=True)

zip_files = ['imgs_part_1.zip', 'imgs_part_2.zip', 'imgs_part_3.zip']

for zip_name in zip_files:
    zip_path = os.path.join(data_path, zip_name)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print(f"Extracted: {zip_name}")

print("\nDone. Contents of extract folder:")
for item in os.listdir(extract_path):
    print(" -", item)

Extracted: imgs_part_1.zip
Extracted: imgs_part_2.zip
Extracted: imgs_part_3.zip

Done. Contents of extract folder:
 - imgs_part_1
 - imgs_part_2
 - imgs_part_3


In [4]:
import os

# Peek inside imgs_part_1 to see the real structure
sample_folder = '/content/images/imgs_part_1'
contents = os.listdir(sample_folder)
print("First 5 items inside imgs_part_1:")
for item in contents[:5]:
    print(" -", item)
print(f"\nTotal items in imgs_part_1: {len(contents)}")

First 5 items inside imgs_part_1:
 - PAT_255_388_77.png
 - PAT_252_383_389.png
 - PAT_282_433_934.png
 - PAT_290_445_988.png
 - PAT_334_703_907.png

Total items in imgs_part_1: 911


In [5]:
import pandas as pd
import os

# Load metadata
csv_path = '/content/drive/MyDrive/skin-cancer-fusion/data/metadata.csv'
df = pd.read_csv(csv_path)

print("Total rows in CSV:", len(df))
print("\nLabel counts (diagnostic column):")
print(df['diagnostic'].value_counts())

# Build a lookup of all image filenames across all 3 folders
image_folders = ['/content/images/imgs_part_1', '/content/images/imgs_part_2', '/content/images/imgs_part_3']

image_lookup = {}  # filename -> full path
for folder in image_folders:
    for fname in os.listdir(folder):
        image_lookup[fname] = os.path.join(folder, fname)

print(f"\nTotal image files found across all folders: {len(image_lookup)}")

# Check each row's img_id against the lookup
found = 0
missing = 0
missing_examples = []

for img_name in df['img_id']:
    if img_name in image_lookup:
        found += 1
    else:
        missing += 1
        if len(missing_examples) < 5:
            missing_examples.append(img_name)

print(f"\nRows with matching image: {found}")
print(f"Rows with missing image: {missing}")
if missing_examples:
    print("Example missing filenames:", missing_examples)

Total rows in CSV: 2298

Label counts (diagnostic column):
diagnostic
BCC    845
ACK    730
NEV    244
SEK    235
SCC    192
MEL     52
Name: count, dtype: int64

Total image files found across all folders: 2298

Rows with matching image: 2298
Rows with missing image: 0


In [6]:
# See all column names
print("All columns:")
print(df.columns.tolist())

# Check missing values per column
print("\nMissing values per column:")
print(df.isnull().sum())

# Look at a few sample rows
df.head()

All columns:
['patient_id', 'lesion_id', 'smoke', 'drink', 'background_father', 'background_mother', 'age', 'pesticide', 'gender', 'skin_cancer_history', 'cancer_history', 'has_piped_water', 'has_sewage_system', 'fitspatrick', 'region', 'diameter_1', 'diameter_2', 'diagnostic', 'itch', 'grew', 'hurt', 'changed', 'bleed', 'elevation', 'img_id', 'biopsed']

Missing values per column:
patient_id               0
lesion_id                0
smoke                  804
drink                  804
background_father      818
background_mother      822
age                      0
pesticide              804
gender                 804
skin_cancer_history    804
cancer_history         804
has_piped_water        804
has_sewage_system      804
fitspatrick            804
region                   0
diameter_1             804
diameter_2             804
diagnostic               0
itch                     0
grew                     0
hurt                     0
changed                  0
bleed                

,patient_id,lesion_id,smoke,drink,background_father,background_mother,age,pesticide,gender,skin_cancer_history,...,diameter_2,diagnostic,itch,grew,hurt,changed,bleed,elevation,img_id,biopsed
0,PAT_1516,1765,NaN,NaN,NaN,NaN,8,NaN,NaN,NaN,...,NaN,NEV,False,False,False,False,False,False,PAT_1516_1765_530.png,False
1,PAT_46,881,False,False,POMERANIA,POMERANIA,55,False,FEMALE,True,...,5.0,BCC,True,True,False,True,True,True,PAT_46_881_939.png,True
2,PAT_1545,1867,NaN,NaN,NaN,NaN,77,NaN,NaN,NaN,...,NaN,ACK,True,False,False,False,False,False,PAT_1545_1867_547.png,False
3,PAT_1989,4061,NaN,NaN,NaN,NaN,75,NaN,NaN,NaN,...,NaN,ACK,True,False,False,False,False,False,PAT_1989_4061_934.png,False
4,PAT_684,1302,False,True,POMERANIA,POMERANIA,79,False,MALE,True,...,5.0,BCC,True,True,False,False,True,True,PAT_684_1302_588.png,True


In [7]:
# Check if the missingness lines up with biopsed status
print("Rows where 'gender' is missing, broken down by biopsed:")
print(df[df['gender'].isnull()]['biopsed'].value_counts())

print("\nRows where 'gender' is NOT missing, broken down by biopsed:")
print(df[df['gender'].notnull()]['biopsed'].value_counts())

# Check the actual unique values in the UNK-style columns
for col in ['itch', 'grew', 'hurt', 'changed', 'bleed', 'elevation']:
    print(f"\nUnique values in '{col}':", df[col].unique())

Rows where 'gender' is missing, broken down by biopsed:
biopsed
False    804
Name: count, dtype: int64

Rows where 'gender' is NOT missing, broken down by biopsed:
biopsed
True     1342
False     152
Name: count, dtype: int64

Unique values in 'itch': ['False' 'True' 'UNK']

Unique values in 'grew': ['False' 'True' 'UNK']

Unique values in 'hurt': ['False' 'True' 'UNK']

Unique values in 'changed': ['False' 'True' 'UNK']

Unique values in 'bleed': ['False' 'True' 'UNK']

Unique values in 'elevation': ['False' 'True' 'UNK']


In [8]:
# Filter to biopsy-confirmed rows with complete metadata
df_clean = df[(df['biopsed'] == True) & (df['gender'].notnull())].copy()

print("Rows in cleaned subset:", len(df_clean))
print("\nLabel counts in cleaned subset:")
print(df_clean['diagnostic'].value_counts())

# Fix the UNK-style columns: convert to a 3-category string (not boolean)
symptom_cols = ['itch', 'grew', 'hurt', 'changed', 'bleed', 'elevation']
for col in symptom_cols:
    print(f"\n{col} value counts in cleaned subset:")
    print(df_clean[col].value_counts())

Rows in cleaned subset: 1342

Label counts in cleaned subset:
diagnostic
BCC    845
SCC    192
ACK    178
NEV     60
MEL     52
SEK     15
Name: count, dtype: int64

itch value counts in cleaned subset:
itch
True     943
False    393
UNK        6
Name: count, dtype: int64

grew value counts in cleaned subset:
grew
True     618
UNK      402
False    322
Name: count, dtype: int64

hurt value counts in cleaned subset:
hurt
False    956
True     376
UNK       10
Name: count, dtype: int64

changed value counts in cleaned subset:
changed
False    778
UNK      396
True     168
Name: count, dtype: int64

bleed value counts in cleaned subset:
bleed
False    780
True     556
UNK        6
Name: count, dtype: int64

elevation value counts in cleaned subset:
elevation
True     1018
False     322
UNK         2
Name: count, dtype: int64


In [10]:
symptom_cols = ['itch', 'grew', 'hurt', 'changed', 'bleed', 'elevation']
for col in symptom_cols:
    df_clean[col] = df_clean[col].astype(str)  # keeps UNK as its own value

In [11]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df_clean, test_size=0.3, stratify=df_clean['diagnostic'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['diagnostic'], random_state=42
)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))
print("\nTrain label counts:")
print(train_df['diagnostic'].value_counts())

Train: 939
Val: 201
Test: 202

Train label counts:
diagnostic
BCC    591
SCC    134
ACK    125
NEV     42
MEL     36
SEK     11
Name: count, dtype: int64


In [12]:
import pandas as pd

# Categorical columns to one-hot encode
cat_cols = ['gender', 'smoke', 'drink', 'pesticide', 'skin_cancer_history',
            'cancer_history', 'itch', 'grew', 'hurt', 'changed', 'bleed', 'elevation',
            'region', 'background_father', 'background_mother']

def encode_metadata(df, cat_cols, numeric_cols, reference_columns=None):
    df_cat = pd.get_dummies(df[cat_cols].astype(str))
    df_num = df[numeric_cols].fillna(df[numeric_cols].median())
    df_encoded = pd.concat([df_num, df_cat], axis=1)
    if reference_columns is not None:
        df_encoded = df_encoded.reindex(columns=reference_columns, fill_value=0)
    return df_encoded

numeric_cols = ['age', 'fitspatrick', 'diameter_1', 'diameter_2']

train_meta = encode_metadata(train_df, cat_cols, numeric_cols)
val_meta = encode_metadata(val_df, cat_cols, numeric_cols, reference_columns=train_meta.columns)
test_meta = encode_metadata(test_df, cat_cols, numeric_cols, reference_columns=train_meta.columns)

print("Feature count:", train_meta.shape[1])
train_meta.head()

Feature count: 70


,age,fitspatrick,diameter_1,diameter_2,gender_FEMALE,gender_MALE,smoke_False,smoke_True,drink_False,drink_True,...,background_mother_FRANCE,background_mother_GERMANY,background_mother_ITALY,background_mother_NETHERLANDS,background_mother_NORWAY,background_mother_POLAND,background_mother_POMERANIA,background_mother_PORTUGAL,background_mother_SPAIN,background_mother_UNK
1364,59,2.0,6.0,4.0,True,False,True,False,True,False,...,False,False,False,False,False,False,True,False,False,False
432,71,2.0,15.0,13.0,False,True,True,False,True,False,...,False,False,False,False,False,False,True,False,False,False
2094,58,3.0,12.0,6.0,False,True,True,False,True,False,...,False,True,False,False,False,False,False,False,False,False
1455,78,1.0,11.0,10.0,False,True,True,False,True,False,...,False,False,False,False,False,False,False,False,False,False
144,70,2.0,11.0,9.0,False,True,True,False,False,True,...,False,True,False,False,False,False,False,False,False,False


In [13]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
le.fit(train_df['diagnostic'])

train_labels = le.transform(train_df['diagnostic'])
val_labels = le.transform(val_df['diagnostic'])
test_labels = le.transform(test_df['diagnostic'])

print("Classes:", le.classes_)
print("Example encoded labels:", train_labels[:5])

Classes: ['ACK' 'BCC' 'MEL' 'NEV' 'SCC' 'SEK']
Example encoded labels: [1 1 1 1 4]


In [14]:
import torch
from torch.utils.data import Dataset
from PIL import Image
from torchvision import transforms
import os

class SkinFusionDataset(Dataset):
    def __init__(self, df, meta_features, labels, image_folders, transform=None):
        self.df = df.reset_index(drop=True)
        self.meta_features = meta_features.reset_index(drop=True)
        self.labels = labels
        self.image_folders = image_folders
        self.transform = transform
        self.image_lookup = {}
        for folder in image_folders:
            for fname in os.listdir(folder):
                self.image_lookup[fname] = os.path.join(folder, fname)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.loc[idx, 'img_id']
        img_path = self.image_lookup[img_name]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        meta = torch.tensor(self.meta_features.iloc[idx].values.astype('float32'))
        label = torch.tensor(self.labels[idx], dtype=torch.long)

        return image, meta, label

In [15]:
image_folders = ['/content/images/imgs_part_1', '/content/images/imgs_part_2', '/content/images/imgs_part_3']

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = SkinFusionDataset(train_df, train_meta, train_labels, image_folders, transform)
val_dataset = SkinFusionDataset(val_df, val_meta, val_labels, image_folders, transform)
test_dataset = SkinFusionDataset(test_df, test_meta, test_labels, image_folders, transform)

# Test one sample
image, meta, label = train_dataset[0]
print("Image shape:", image.shape)
print("Meta shape:", meta.shape)
print("Label:", label)

Image shape: torch.Size([3, 224, 224])
Meta shape: torch.Size([70])
Label: tensor(1)


In [16]:
from torch.utils.data import DataLoader

batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Test one batch
images, metas, labels = next(iter(train_loader))
print("Batch image shape:", images.shape)
print("Batch meta shape:", metas.shape)
print("Batch labels shape:", labels.shape)

Batch image shape: torch.Size([32, 3, 224, 224])
Batch meta shape: torch.Size([32, 70])
Batch labels shape: torch.Size([32])


In [17]:
import torch.nn as nn
import torchvision.models as models

class FusionModel(nn.Module):
    def __init__(self, meta_input_dim, num_classes):
        super().__init__()

        # Image branch: pretrained ResNet18
        resnet = models.resnet18(weights='IMAGENET1K_V1')
        self.image_branch = nn.Sequential(*list(resnet.children())[:-1])  # remove final FC
        self.image_out_dim = 512

        # Metadata branch: simple MLP
        self.meta_branch = nn.Sequential(
            nn.Linear(meta_input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU()
        )
        self.meta_out_dim = 32

        # Fusion + classifier
        self.classifier = nn.Sequential(
            nn.Linear(self.image_out_dim + self.meta_out_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, image, meta):
        img_feat = self.image_branch(image).flatten(1)
        meta_feat = self.meta_branch(meta)
        combined = torch.cat([img_feat, meta_feat], dim=1)
        return self.classifier(combined)

In [18]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

model = FusionModel(meta_input_dim=70, num_classes=6).to(device)
print(model)

Using device: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 75.6MB/s]


FusionModel(
  (image_branch): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_st

In [19]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

print("Class weights:", class_weights)

Class weights: tensor([ 1.2520,  0.2648,  4.3472,  3.7262,  1.1679, 14.2273], device='cuda:0')


In [20]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for images, metas, labels in loader:
        images, metas, labels = images.to(device), metas.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images, metas)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / len(loader), correct / total

In [21]:
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, metas, labels in loader:
            images, metas, labels = images.to(device), metas.to(device), labels.to(device)
            outputs = model(images, metas)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return total_loss / len(loader), correct / total

In [22]:
num_epochs = 15

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    print(f"Epoch {epoch+1}/{num_epochs} - Train loss: {train_loss:.4f}, Train acc: {train_acc:.4f} - Val loss: {val_loss:.4f}, Val acc: {val_acc:.4f}")

Epoch 1/15 - Train loss: 1.7269, Train acc: 0.2684 - Val loss: 1.5644, Val acc: 0.3980
Epoch 2/15 - Train loss: 1.2518, Train acc: 0.4654 - Val loss: 1.5083, Val acc: 0.4527
Epoch 3/15 - Train loss: 0.8602, Train acc: 0.6688 - Val loss: 1.3683, Val acc: 0.4428
Epoch 4/15 - Train loss: 0.5175, Train acc: 0.7955 - Val loss: 1.4866, Val acc: 0.5672
Epoch 5/15 - Train loss: 0.2909, Train acc: 0.9382 - Val loss: 1.5758, Val acc: 0.5672
Epoch 6/15 - Train loss: 0.1720, Train acc: 0.9659 - Val loss: 1.6203, Val acc: 0.5821
Epoch 7/15 - Train loss: 0.1042, Train acc: 0.9894 - Val loss: 1.7378, Val acc: 0.6368
Epoch 8/15 - Train loss: 0.0822, Train acc: 0.9872 - Val loss: 1.7460, Val acc: 0.6219
Epoch 9/15 - Train loss: 0.0539, Train acc: 0.9915 - Val loss: 1.9334, Val acc: 0.6667
Epoch 10/15 - Train loss: 0.0465, Train acc: 0.9925 - Val loss: 1.9300, Val acc: 0.6318
Epoch 11/15 - Train loss: 0.0468, Train acc: 0.9904 - Val loss: 1.9824, Val acc: 0.6368
Epoch 12/15 - Train loss: 0.0465, Train a